# Transformer Decoder

## Code

In [ ]:
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn
import models.deep_learning.components as comp
from helpers import get_device


## Testing

In [ ]:
# input parameters
N = 3
M = 4
batch_size = 2
device = get_device()
dtype = torch.float32

# Decoder Layer parameters
d_model = 4
nhead = 2
dim_feedforward = 64
dropout = 0.2
layer_norm_eps = 1e-5
norm_first = True
bias = True

src_mask = comp.create_random_mask((N, N), device=device)
tgt_mask = comp.create_causal_mask((M, M), device=device)
memory_mask = comp.create_random_mask((M, N), device=device)


## Transformer decoder parameters
num_enc_layers = 3
num_dec_layers = 5
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [ ]:
torch.manual_seed(0)
x = torch.randn(batch_size, N, d_model, device=device, dtype=dtype)
tgt = torch.randn(batch_size, M, d_model, device=device, dtype=dtype)

In [ ]:
init_seed = 42  # avoide weights initialization randomness effects
train_seed = 24  # avoid dropout randomness effects

In [ ]:
torch.manual_seed(init_seed)
tf = mynn.Transformer(
    d_model,
    nhead,
    num_encoder_layers=num_enc_layers,
    num_decoder_layers=num_dec_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation_cls=nn.GELU,
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)

torch.manual_seed(init_seed)

nn_tf = nn.Transformer(
    d_model,
    nhead,
    num_encoder_layers=num_enc_layers,
    num_decoder_layers=num_dec_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation="gelu",
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
    batch_first=True,
)

tf.load_weights_from_torch_transformer(nn_tf)

### Evaluation

In [ ]:
tf.eval()
tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

In [ ]:
nn_tf.eval()
nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

### Training

In [ ]:
mse = torch.nn.MSELoss()

In [ ]:
torch.manual_seed(train_seed)
nn_tf.train()
out = nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)
print(out)
loss = mse(out, tgt)
loss.backward()
optimizer = torch.optim.SGD(nn_tf.parameters(), lr=1e-3)
optimizer.step()
nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

In [ ]:
torch.manual_seed(train_seed)
tf.train()
out = tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)
print(out)
loss = mse(out, tgt)
loss.backward()
optimizer = torch.optim.SGD(tf.parameters(), lr=1e-3)
optimizer.step()
tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)